<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/Embedding_Model_FT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#간단한 구현 예시

In [ ]:
원래 있는 모델에 새로운 데이터 fit하면 끝.
dataloader : pytorch에서 dataset으로부터 data불러오는애.
train_loss : 손실함수
warmup_steps : 예열스텝

In [10]:
from sentence_transformers import SentenceTransformer, losses, InputExample
from torch.utils.data import DataLoader

model = SentenceTransformer('distilbert-base-multilingual-cased')

train = [
    InputExample(texts=["AI란 무엇인가?", "AI는 인간의 지능을 모방한 기술입니다."]),
    InputExample(texts=["딥러닝이란?", "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다."]),
    InputExample(texts=["Python은 어디에 쓰이나요?", "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다."]),
    InputExample(texts=["자연어 처리란?", "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다."])
]


train_dataloader = DataLoader(train, shuffle=True, batch_size = 32)
train_loss = losses.MultipleNegativesRankingLoss(model, scale=20.0)
warmup_steps = int(len(train_dataloader) * 0.1)

model.fit(
    train_objectives = [(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps = warmup_steps,
    optimizer_params = {'lr' : 2e-5},
    output_path = './FT_model'
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
import torch
from sentence_transformers import SentenceTransformer, util

# 1. 학습된 모델 로드 (저장된 경로)
model_path = 'FT_model'
model = SentenceTransformer(model_path)

# 2. 지식 베이스(Knowledge Base) 정의
# 모델이 답변할 수 있는 '정답' 리스트입니다.
answers = [
  "AI는 인간의 지능을 모방한 기술입니다.",
  "신경망을 여러 층 쌓아 데이터로부터 학습하는 기계학습 방법입니다.",
  "Python은 데이터 분석, 웹 개발, AI 등에 널리 사용됩니다.",
  "컴퓨터가 인간의 언어를 이해하고 처리하는 AI의 한 분야입니다."
]

# 3. 답변 후보들을 미리 임베딩(벡터화) 해둡니다.
answer_embeddings = model.encode(answers, convert_to_tensor=True)

def chat():
    print("AI 챗봇과 대화를 시작합니다! (종료하려면 'exit' 입력)")

    while True:
        # 사용자 입력
        user_input = input("질문: ")
        if user_input.lower() == 'exit':
            break

        # 4. 사용자 질문 임베딩
        question_embedding = model.encode(user_input, convert_to_tensor=True)

        # 5. 코사인 유사도 계산 (질문과 답변 후보들 비교)
        # util.cos_sim은 [질문 개수 x 답변 개수]의 유사도 행렬을 반환합니다.
        cos_scores = util.cos_sim(question_embedding, answer_embeddings)[0]

        # 6. 가장 유사도가 높은 답변의 인덱스 추출
        top_result_idx = torch.argmax(cos_scores).item()
        confidence = cos_scores[top_result_idx].item()

        # 7. 결과 출력 (유사도가 너무 낮으면 모른다고 답변하게 설정 가능)
        if confidence > 0.5:  # 임계값 설정
            print(f"답변: {answers[top_result_idx]} (신뢰도: {confidence:.2f})")
        else:
            print("답변: 죄송합니다, 관련 정보를 찾을 수 없습니다.")
        print("-" * 30)

if __name__ == "__main__":
    chat()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

AI 챗봇과 대화를 시작합니다! (종료하려면 'exit' 입력)


KeyboardInterrupt: Interrupted by user

#실제 예시

## 환경설정

In [14]:
!pip install PyPDF2 datasets sentence-transformers==3.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.3.0
    Uninstalling sentence-transformers-5.3.0:
      Successfully uninstalled sentence-transformers-5.3.0


In [1]:
import os
import requests
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import torch
from sklearn.metrics.pairwise import cosine_similarity

## 데이터 다운로드

In [3]:
urls = [
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_japan_2024.pdf",
   "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_usa_2024.pdf"
]

for url in urls:
   filename = url.split("/")[-1]
   response = requests.get(url)
   with open(filename, "wb") as f:
       f.write(response.content)
   print(f"{filename} 다운로드 완료")

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료


pdf to txt

In [4]:
import PyPDF2

def extract_text_from_pdf(pdf_path):
   """PDF 파일에서 텍스트를 추출하는 함수"""
   text_chunks = []
   with open(pdf_path, 'rb') as file:
       pdf_reader = PyPDF2.PdfReader(file)
       for page_num in range(len(pdf_reader.pages)):
           page = pdf_reader.pages[page_num]
           text = page.extract_text()
           # 페이지 단위로 청크 생성
           if text.strip():
               text = text.strip()
               # 문서 길이가 10자 초과인 경우만 추가
               if len(text) > 10:
                   text_chunks.append(text)
   return text_chunks

# 미국 ICT 동향 (학습 데이터)
train_corpus = extract_text_from_pdf('ict_usa_2024.pdf')
print(f'학습 데이터 문서 개수: {len(train_corpus)}')

# 일본 ICT 동향 (검증 데이터)
val_corpus = extract_text_from_pdf('ict_japan_2024.pdf')
print(f'검증 데이터 문서 개수: {len(val_corpus)}')

학습 데이터 문서 개수: 26
검증 데이터 문서 개수: 27


## 데이터 정제

데이터 이용해 [질문-페이지 텍스트] 쌍 만들기

In [17]:
os.environ['OPENAI_API_KEY'] = ''

In [18]:
# OpenAI 클라이언트 초기화
client = OpenAI()

# 3. 각 문서에 대한 질문 생성 (OpenAI API 사용)
def generate_queries(corpus, num_questions_per_chunk=2):
    all_queries = []
    all_positive_docs = []

    # 기본 프롬프트 템플릿 설정
    prompt_template = """\
    다음은 참고할 내용입니다.

    ---------------------
    {context_str}
    ---------------------

    위 내용을 바탕으로 낼 수 있는 질문을 {num_questions_per_chunk}개 만들어주세요.
    질문만 작성하고 실제 정답이나 보기 등은 작성하지 않습니다.

    해당 질문은 본문을 볼 수 없다고 가정합니다.
    따라서 '위 본문을 바탕으로~' 라는 식의 질문은 할 수 없습니다.

    질문은 아래와 같은 형식으로 번호를 나열하여 생성하십시오.

    1. (질문)
    2. (질문)
    """

    # corpus의 각 문서에 대해 반복 실행
    for text in tqdm(corpus):
        # 현재 문서에 대한 프롬프트 생성
        messages = [
            {"role": "system", "content": "You are a helpful assistant that generates questions based on provided content."},
            {"role": "user", "content": prompt_template.format(
                context_str=text,
                num_questions_per_chunk=num_questions_per_chunk
            )}
        ]

        # GPT 모델을 사용해 질문 생성
        response = client.chat.completions.create(
            model="gpt-5.4-nano",
            messages=messages,
            temperature=0.7,
        )

        # 응답을 줄바꿈을 기준으로 분리하여 개별 질문으로 만듦
        result = response.choices[0].message.content.strip().split("\n")

        # 질문 형식 정리
        questions = []
        for line in result:
            if line.strip():
                parts = line.strip().split('. ', 1)
                if len(parts) > 1:
                    questions.append(parts[1])
                else:
                    questions.append(parts[0])

        # 빈 질문 제거
        questions = [q for q in questions if len(q) > 0]

        # 각 질문에 대해 문서 매칭 및 저장
        for question in questions:
            all_queries.append(question)
            all_positive_docs.append(text)

    return all_queries, all_positive_docs

In [19]:
train_queries, train_positive_docs = generate_queries(train_corpus)
val_queries, val_positive_docs = generate_queries(val_corpus)

  0%|          | 0/26 [00:00<?, ?it/s]

  0%|          | 0/27 [00:00<?, ?it/s]

In [26]:
#InputExample 형식으로 변환
train_examples = []
for q, d in zip(train_queries, train_positive_docs):
  example = InputExample(texts = [q, d])
  train_examples.append(example)

In [24]:
for i in [0, 1, 10, 20]:
  print(train_queries[i])
  print(train_positive_docs[i])
  print('=' * 100)

미국의 인공지능 챗봇 개발 동향과 관련된 핵심 내용을 고르시오.
Ⅰ ICT국가산업현황  4
(*) SUMMARY
1. 국가 개황
2. ICT 정부기구
3. ICT 주요정책
4. ICT 주요법령및규제
5. ICT 주요기업
6. 한국 협력 및 국내기업 진출사례
Ⅱ ICT이슈Top 10  16
(*) SUMMARY
① 미국 빅테크 기업, 인공지능 챗봇 개발에 주력
② 미국, 일본과 양자컴퓨팅 개발 협력
③ 미국, 우주 클라우드 컴퓨팅 시장 주도
④ 미국, 드론 배송 도입 활발
⑤ 미국, 긍정적인 의료 AI 인식 바탕으로 연구 활발
⑥ 미국, 반도체 산업 활성화에 박차
⑦ 미국, 기술 교류를 위한 국가 간 협력 활발
⑧ 미국, 사이버 보안 대응 강화
⑨ 미국, 6G 주도권 확보 위한 연구 추진
⑩ 미 국방부 , 디지털 트윈 기술 도입 확대
※ 참고문헌
미국이 디지털 트윈 기술을 확대 도입하는 주체와 목적(또는 활용 분야)에 해당하는 내용을 서술하시오.
Ⅰ ICT국가산업현황  4
(*) SUMMARY
1. 국가 개황
2. ICT 정부기구
3. ICT 주요정책
4. ICT 주요법령및규제
5. ICT 주요기업
6. 한국 협력 및 국내기업 진출사례
Ⅱ ICT이슈Top 10  16
(*) SUMMARY
① 미국 빅테크 기업, 인공지능 챗봇 개발에 주력
② 미국, 일본과 양자컴퓨팅 개발 협력
③ 미국, 우주 클라우드 컴퓨팅 시장 주도
④ 미국, 드론 배송 도입 활발
⑤ 미국, 긍정적인 의료 AI 인식 바탕으로 연구 활발
⑥ 미국, 반도체 산업 활성화에 박차
⑦ 미국, 기술 교류를 위한 국가 간 협력 활발
⑧ 미국, 사이버 보안 대응 강화
⑨ 미국, 6G 주도권 확보 위한 연구 추진
⑩ 미 국방부 , 디지털 트윈 기술 도입 확대
※ 참고문헌
미국 국가과학기술위원회(NSTC)가 1993년에 설립되었으며 백악관 산하 자문기구로서 수행하는 주요 기능 5가지는 무엇인가?
8 Ⅰ. ICT 국가 산업 현황
 2.ICT 정부기구
 ③ 국가과학기술위원회 (NSTC) 
 미국 국가과학기술위원회 

## 학습 진행

In [34]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')


train_dataloader = DataLoader(train_examples, shuffle=True, batch_size = 32)
train_loss = losses.MultipleNegativesRankingLoss(model, scale=20.0)
warmup_steps = int(len(train_dataloader) * 0.1)

model.fit(
    train_objectives = [(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps = warmup_steps,
    optimizer_params = {'lr' : 2e-5},
    output_path = './FT_model'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


##평가 진행

검증 데이터셋 만들기

In [46]:
val_dataset = {
    'queries': {},
    'corpus': {},
    'relevant_docs': {}
}

# 문서 ID를 먼저 생성
doc_ids = {}
for i, doc in enumerate(val_corpus):
    doc_id = f"d{i}"
    val_dataset['corpus'][doc_id] = doc
    doc_ids[doc] = doc_id

# 질문에 ID 부여하고 관련 문서 설정
for i, (query, doc) in enumerate(zip(val_queries, val_positive_docs)):
    query_id = f"q{i}"
    val_dataset['queries'][query_id] = query

    # 이 질문이 어떤 문서에서 왔는지 찾기
    doc_id = doc_ids[doc]

    # 관련 문서 설정
    if query_id not in val_dataset['relevant_docs']:
        val_dataset['relevant_docs'][query_id] = set()
    val_dataset['relevant_docs'][query_id].add(doc_id)


In [58]:
val_dataset['queries']

{'q0': '일본의 ICT 산업 발전 방향을 보여주는 ‘데이터센터 허브’와 관련된 내용으로부터, 일본이 아시아에서 차지하는 위치는 무엇인가?',
 'q1': '일본의 ICT 주요 이슈 중 ‘정부 행정 업무에 생성형 AI 도입’과 ‘디지털 인력 강화’가 함께 언급될 때, 두 이슈가 시사하는 정책적 목표는 무엇인가?',
 'q2': '한국의 ICT 국가 산업 현황에서 다루는 항목(국가 개황, ICT 정부기구, ICT 주요 정책, ICT 주요 법령 및 규제, ICT 주요기업, 한국 협력 및 국내기업 진출사례) 각각이 의미하는 바를 간단히 설명하시오.',
 'q3': 'ICT 주요 법령 및 규제가 ICT 산업 발전에 미치는 영향을 평가할 때, 어떤 관점(예: 규제 목적, 적용 범위, 산업/기업에 대한 영향)을 기준으로 정리할 수 있는지 서술하시오.',
 'q4': '일본의 인터넷 사용자 비중과 고정 광대역 가입자 비중은 각각 얼마인가?',
 'q5': '일본의 글로벌 혁신지수에서 ‘창조적 생산’ 지표 순위가 ‘인프라’ 및 ‘지식 및 기술 생산’ 지표에 비해 상대적으로 낮게 나타나는 이유(또는 특징)를 설명하시오.',
 'q6': '일본의 총무성(MIC)이 담당하는 ICT 관련 주요 역할 5가지는 무엇인가?',
 'q7': '일본 총무성이 최근 추진한 로컬 5G 및 텔레워크 실증 사업과 관련된 구체적인 내용은 무엇인가?',
 'q8': '일본 문부과학성(MEXT)이 ICT 분야에서 수행하는 역할로 제시된 항목들을 모두 고르시오.',
 'q9': '문부과학성이 최근 추진한 교육 부문 디지털 전환(DX) 관련 정책과 이를 위해 마련한 기금의 규모 및 지원 대상(기관/단위)은 무엇인지 설명하시오.',
 'q10': '일본의 경제산업성(METI)이 경제활력을 위한 정책 수립·실행과 함께 담당하는 주요 업무로 무엇이 포함되는가?',
 'q11': '경제산업성(METI)이 최근 추진·발표한 내용 중, 서비스 로봇 안전 국제 표준 발표, 일본-EU EPA의 ‘데이터의 자유로운 흐

In [60]:
val_dataset['corpus']

{'d0': 'Ⅰ ICT국가산업현황  4\n(*) SUMMARY\n1. 국가 개황\n2. ICT 정부기구\n3. ICT 주요정책\n4. ICT 주요법령및규제\n5. ICT 주요기업\n6. 한국 협력 및 국내기업 진출사례\nⅡ ICT이슈Top 10  16\n(*) SUMMARY\n① 일본, 아시아에서 두 번째로 큰 데이터센터 허브\n② 일본, 자체 개발 소프트웨어로 사이버보안 강화\n③ 일본, Web3 산업 성장 촉진 도모\n④ 일본, 정부 행정 업무에 생성형 AI 도입\n⑤ 일본, 첫 자체 제작 양자컴퓨터 공개\n⑥ 일본, 6G 기술 강화 위해 협력 및 규제 완화\n⑦ 일본, 레벨 4 자율주행 허용\n⑧ 일본, 생체인식 결제 도입 증가\n⑨ 일본, 행정 서비스 디지털화 노력\n⑩ 일본, 인재 부족으로 디지털 인력 강화에 힘써\n※ 참고문헌',
 'd1': 'Ⅰ ICT 국가 산업 현황                   4\n   (*) SUMMARY\n   1. 국가 개황\n   2. ICT 정부기구\n   3. ICT 주요 정책\n   4. ICT 주요 법령 및 규제\n   5. ICT 주요기업\n   6. 한국 협력 및 국내기업 진출사례',
 'd2': '5 Ⅰ. ICT 국가 산업 현황\n 1.국가 개황\n 일본, 글로벌 혁신지수 세계 40위\n• 일본의 인터넷 사용자 비중은 82.9% 이며 고정 광대역 가입자 비중은 36.0% 임\n• 일본 글로벌 혁신지수는 세계 13위임. ‘인프라 ’ 및 ‘지식 및 기술 생산’ 지표가 상대적으로 \n우위에 있으며 , ‘창조적 생산’이 상대적 열위를 보임\n 기시다 총리, 결제 활성화 정책에 초점\n• 2023년 일본 물가상승률은 3.1%로 41년에 최대폭을 기록함 . 일본은행은 2023년 12월 \n금융정책결정회의에서 단기금리 목표를 △0.1%로 동결하면서 마이너스 금리 종료를 보류함\n• 일본 기시다 후미오 (岸⽥⽂雄 ) 총리는 향후 경제 활성화를 위한 정책에 초점을 맞출 것임을 밝힘. \n그중에서도 감세 논의에 속도를 

In [61]:
val_dataset['relevant_docs']

{'q0': {'d0'},
 'q1': {'d0'},
 'q2': {'d1'},
 'q3': {'d1'},
 'q4': {'d2'},
 'q5': {'d2'},
 'q6': {'d3'},
 'q7': {'d3'},
 'q8': {'d4'},
 'q9': {'d4'},
 'q10': {'d5'},
 'q11': {'d5'},
 'q12': {'d6'},
 'q13': {'d6'},
 'q14': {'d7'},
 'q15': {'d7'},
 'q16': {'d8'},
 'q17': {'d8'},
 'q18': {'d9'},
 'q19': {'d9'},
 'q20': {'d10'},
 'q21': {'d10'},
 'q22': {'d11'},
 'q23': {'d11'},
 'q24': {'d12'},
 'q25': {'d12'},
 'q26': {'d13'},
 'q27': {'d13'},
 'q28': {'d14'},
 'q29': {'d14'},
 'q30': {'d15'},
 'q31': {'d15'},
 'q32': {'d16'},
 'q33': {'d16'},
 'q34': {'d17'},
 'q35': {'d17'},
 'q36': {'d18'},
 'q37': {'d18'},
 'q38': {'d19'},
 'q39': {'d19'},
 'q40': {'d20'},
 'q41': {'d20'},
 'q42': {'d21'},
 'q43': {'d21'},
 'q44': {'d22'},
 'q45': {'d22'},
 'q46': {'d23'},
 'q47': {'d23'},
 'q48': {'d24'},
 'q49': {'d24'},
 'q50': {'d25'},
 'q51': {'d25'},
 'q52': {'d26'},
 'q53': {'d26'}}

In [62]:
# 검증 데이터셋에서 코퍼스(전체 문서), 쿼리, 그리고 각 쿼리와 관련된 문서 가져오기
corpus = val_dataset['corpus']  # 검색 대상 문서
queries = val_dataset['queries']  # 검색어(쿼리)
relevant_docs = val_dataset['relevant_docs']  # 각 쿼리와 관련된 문서 (포지티브)

# Information Retrieval 평가 도구 설정: 쿼리-문서 검색 성능 평가
evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs)

In [68]:
def evaluate(model_id, evaluator):
  os.makedirs('evaluate_results', exist_ok = True)

  model = SentenceTransformer(model_id)
  result = evaluator(model)

  return result

In [69]:
#basemodel 성능측정
basemodel_evaluate_res = evaluate('distilbert-base-nli-stsb-mean-tokens', evaluator)
ftmodel_evaluate_res = evaluate('FT_model', evaluator)

In [81]:
df = pd.DataFrame([basemodel_evaluate_res, ftmodel_evaluate_res])
df.index = ['base','ft']
df

,cosine_accuracy@1,cosine_accuracy@3,cosine_accuracy@5,cosine_accuracy@10,cosine_precision@1,cosine_precision@3,cosine_precision@5,cosine_precision@10,cosine_recall@1,cosine_recall@3,cosine_recall@5,cosine_recall@10,cosine_ndcg@10,cosine_mrr@10,cosine_map@100
base,0.111111,0.277778,0.370370,0.537037,0.111111,0.092593,0.074074,0.053704,0.111111,0.277778,0.370370,0.537037,0.296666,0.223038,0.250809
ft,0.092593,0.314815,0.407407,0.537037,0.092593,0.104938,0.081481,0.053704,0.092593,0.314815,0.407407,0.537037,0.304465,0.231577,0.259346
